# Linux Installation: Architecture, Partitioning, Filesystems, Swap, and Setup (Educational Notebook)
This notebook walks through what happens *before* and *during* a Linux installation: choosing the right image for your hardware, laying out disk partitions, picking a filesystem, sizing swap, and running through the install itself.

## 1. Planning an Installation

Before installing anything, a few questions need answers:

- **What hardware architecture is the target machine?** Most PCs are `x86_64` (also called `amd64`); many phones, single-board computers (like the Raspberry Pi), and newer laptops use `aarch64`/`arm64`. The installation image must match the CPU architecture.
- **BIOS or UEFI firmware?** Modern machines almost always boot via **UEFI**, which affects both the partition table type and the bootloader setup used later. Older or legacy-mode systems still use **BIOS** (also called legacy boot).
- **Where will the installer come from?** Typically a downloaded `.iso` image written to a USB drive, or a network/PXE boot in server environments.
- **Verify the download** - distributions publish checksums (`sha256sum`) and often a GPG signature for the ISO, so you can confirm the file wasn't corrupted or tampered with before booting from it.

```bash
sha256sum distro.iso                 # compute a checksum to compare against the published one
uname -m                               # print this machine's architecture (e.g. x86_64)
```


## 2. Firmware and Partition Table: BIOS/MBR vs. UEFI/GPT

The system's firmware type determines which partition table format the disk needs:

| | BIOS (legacy) | UEFI (modern) |
|---|---|---|
| Partition table | MBR (Master Boot Record) | GPT (GUID Partition Table) |
| Max partitions | 4 primary (or 3 primary + 1 extended holding logical partitions) | 128 by default, no primary/extended distinction |
| Max disk size | 2 TiB | Effectively unlimited (exabytes) |
| Boot mechanism | Bootloader code lives in the MBR's first sector | A dedicated **EFI System Partition (ESP)**, formatted FAT32, holds bootloader `.efi` files |
| Redundancy | None - a damaged MBR can make the disk unbootable | GPT stores a backup header/table at the end of the disk |

On a UEFI system, the installer creates (or expects) a small FAT32 **EFI System Partition** (commonly mounted at `/boot/efi`), in addition to the Linux partitions themselves.

```bash
lsblk -o NAME,SIZE,FSTYPE,MOUNTPOINT     # list disks and partitions with their filesystem/mountpoint
parted -l                                  # show each disk's partition table type (msdos/gpt) and layout
```


## 3. Disk Partitioning

A disk is divided into **partitions** so that different parts of the filesystem hierarchy, or different operating systems, can live in separate, independently-manageable regions of the same physical disk.

A common Linux partition layout looks like:

```
/boot/efi   (or /boot on BIOS systems)   -  bootloader files
/                                          -  the root filesystem (everything else, if not split further)
/home                                      -  user data, optionally on its own partition
swap                                       -  swap space (see below)
```

Separating `/home` onto its own partition is a common practice: it lets you reinstall or reformat the root filesystem without touching user data.

```bash
fdisk /dev/sda        # interactive MBR/GPT partition editor (classic, keyboard-driven)
parted /dev/sda         # more modern partition editor, supports scripting
mkfs.ext4 /dev/sda1       # format a partition with a filesystem, after partitioning it
```

Partitioning is destructive to any existing data on the affected region of the disk - always double-check the target device name (e.g. `/dev/sda` vs `/dev/sdb`) before writing changes.


## 4. Linux Filesystems

Once a partition exists, it needs to be **formatted** with a filesystem before Linux can store files on it.

| Filesystem | Notes |
|---|---|
| `ext4` | The long-standing default on most distributions; mature, reliable, journaling. |
| `xfs` | High-performance journaling filesystem, the default on RHEL/Fedora; handles very large files well. |
| `btrfs` | Copy-on-write filesystem with built-in snapshots and volume management; the default on some distributions (e.g. openSUSE). |
| `vfat`/`fat32` | Used for the EFI System Partition, since UEFI firmware requires FAT for `/boot/efi`. |
| `swap` | Not a general-purpose filesystem - a special format used only for swap space (see below). |

```bash
mkfs.ext4 /dev/sda2      # format a partition as ext4
mkfs.xfs /dev/sda2         # format a partition as xfs
blkid                        # show the filesystem type and UUID of every partition
df -hT                         # show mounted filesystems, their type, and free space
```


## 5. Swap Space

**Swap** is disk space the kernel can use as an overflow for RAM: when physical memory is full, the kernel can move ("swap out") memory pages belonging to idle processes to disk, freeing RAM for active work. Swap is much slower than RAM, so heavy reliance on it will noticeably slow a system down - it's a safety margin, not a substitute for enough RAM.

Swap can be either a **dedicated swap partition** or a **swap file** on an existing filesystem; modern distributions frequently default to a swap file for flexibility (it can be resized without repartitioning).

Rough sizing guidelines (these vary by workload and by whether the system uses hibernation):
- Little or no swap on servers with plenty of RAM and no hibernation requirement.
- Roughly equal to RAM, or a bit more, if the system needs to **hibernate** (suspend-to-disk), since the whole RAM contents must fit in swap.
- A modest fixed amount (e.g. 2-4 GiB) as a general safety margin on desktop systems.

```bash
mkswap /dev/sda3          # initialize a partition as swap
swapon /dev/sda3            # activate a swap partition
swapon --show                 # list active swap devices/files
free -h                         # show memory and swap usage
```


## 6. The Installation Process

With the hardware, partition table, filesystems, and swap decided, the actual install typically follows this sequence:

1. **Boot the installer** from USB/DVD/network and select language/keyboard layout.
2. **Partition the target disk** - either let the installer do it automatically, or lay it out manually as planned above.
3. **Select and format filesystems**, and choose mount points for each partition.
4. **Choose software** to install (minimal base system, desktop environment, server roles, etc.).
5. **Install the bootloader** (commonly GRUB on BIOS/MBR and BIOS-compatible UEFI systems) so the firmware can find and start the new system - this is where the EFI System Partition from step 2/3 gets used on UEFI machines.
6. **Create the initial user account** and set the root/administrator password.
7. **Reboot into the new system** and remove the installation media.
8. **Post-install setup** - update the package index and installed packages, configure networking, and install any additional needed software.

```bash
# a typical first-boot checklist, distro-family dependent
sudo apt update && sudo apt upgrade      # Debian/Ubuntu family
sudo dnf upgrade --refresh                 # Fedora/RHEL family
```


## 7. Verifying the Installation

After the first boot, a few quick checks confirm the install went as planned:

```bash
lsblk -o NAME,SIZE,FSTYPE,MOUNTPOINT     # confirm partitions mounted where expected
df -hT                                     # confirm filesystem types and free space
swapon --show                                # confirm swap is active
cat /etc/fstab                                 # confirm the persistent mount configuration matches the layout
```

`/etc/fstab` is the file that tells the system which filesystems to mount automatically at boot - it should list every partition set up during installation, along with its mount point, filesystem type, and mount options.


## Hands-on

If you have access to a spare disk or a virtual machine, try (in a VM, never on a disk with data you care about):

```bash
lsblk                       # identify available disks
sudo fdisk /dev/sdb           # (example) create a couple of partitions on a scratch disk
sudo mkfs.ext4 /dev/sdb1        # format one as ext4
sudo mkswap /dev/sdb2             # initialize the other as swap
sudo swapon /dev/sdb2               # activate it
free -h                               # confirm swap is now visible
sudo swapoff /dev/sdb2                  # deactivate it again when done
```


## Review Questions

1. What determines whether a disk should use an MBR or a GPT partition table?
2. What is the EFI System Partition, what filesystem must it use, and why?
3. Why might you put `/home` on its own partition, separate from `/`?
4. Name two Linux filesystems and one situation where you might prefer one over the other.
5. What is swap space used for, and why is relying heavily on it slower than just having more RAM?
6. Why does a system that needs to hibernate typically need swap at least as large as its RAM?
7. Which file records the persistent mount configuration that gets used automatically at every boot?
8. List, in order, the major steps of a typical Linux installation from booting the installer to first boot.


# Cheat Sheet

```
Architecture/firmware check:
  uname -m               CPU architecture
  ls /sys/firmware/efi     (exists only if booted via UEFI)

Partition tables:
  BIOS -> MBR   (4 primary partitions, 2 TiB limit)
  UEFI -> GPT   (128+ partitions, huge size limit, needs a FAT32 ESP)

Partitioning & formatting:
  fdisk /dev/sdX | parted /dev/sdX     partition a disk
  mkfs.ext4 /dev/sdX1                    format as ext4
  mkfs.xfs /dev/sdX1                       format as xfs
  blkid                                      show filesystem types/UUIDs

Swap:
  mkswap /dev/sdX2   ->   swapon /dev/sdX2   ->   swapon --show   ->   swapoff /dev/sdX2

Inspecting the result:
  lsblk -o NAME,SIZE,FSTYPE,MOUNTPOINT
  df -hT
  cat /etc/fstab
```
